# CDV Pipeline — GPU Worker (Google Colab)

This notebook turns your Colab session into a GPU worker that polls Supabase for `pending` jobs and processes them.

### Quick Start
1. Set **Runtime → Change runtime type → T4 GPU**
2. Add your secrets in the 🔑 **Secrets** panel (left sidebar):
   - `SUPABASE_URL`
   - `SUPABASE_SERVICE_ROLE_KEY`
   - `GITHUB_TOKEN` (Required for private repo: access to 'Contents' and 'Metadata')
3. Run all cells (Runtime → Run all)

In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
DRIVE_WORK_DIR = '/content/drive/MyDrive/CDV_WorkDir'
os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
print(f'Full-res files will persist to: {DRIVE_WORK_DIR}')

In [ ]:
# @title 2. Install Node.js 20 & pnpm
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - 
!apt-get install -y nodejs
!npm install -g pnpm

print('\nNode.js version:')
!node -v
print('pnpm version:')
!pnpm -v

In [ ]:
# @title 3. Install system & Python dependencies
!apt-get update -qq && apt-get install -y ffmpeg curl > /dev/null 2>&1
!pip install -q gdown faster-whisper

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                       capture_output=True, text=True)
if result.returncode == 0:
    print(f'GPU detected: {result.stdout.strip()}')
else:
    print('WARNING: No GPU detected. Change runtime type to T4 GPU.')

In [ ]:
# @title 4. Clone repo and install Node deps
from google.colab import userdata
import os

REPO_NAME = 'cdv-yt-clipper'
REPO_OWNER = 'PGA-LaserG'
REPO_DIR = '/content/CDV'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or ''

if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
else:
    REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_OWNER}/{REPO_NAME} into {REPO_DIR}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Pulling latest changes in {REPO_DIR}...")
    !git -C {REPO_DIR} pull

if os.path.exists(os.path.join(REPO_DIR, 'package.json')):
    %cd {REPO_DIR}
    print("Installing Node dependencies (this may take 1-2 minutes)...")
    !pnpm install --no-frozen-lockfile
    print('\nRepo ready.')
else:
    print(f'CRITICAL ERROR: Failed to clone repository correctly. Directory {REPO_DIR} is missing or empty.')
    !ls -la /content

In [ ]:
# @title 5. Configure Environment
from google.colab import userdata
import os

os.environ['SUPABASE_URL']               = userdata.get('SUPABASE_URL')
os.environ['SUPABASE_SERVICE_ROLE_KEY']  = userdata.get('SUPABASE_SERVICE_ROLE_KEY')
os.environ['PIPELINE_MODE']              = 'cloud_limited'
os.environ['WORKER_ID']                  = 'colab-gpu-worker-01'
os.environ['COLAB_GPU']                  = 'true'
os.environ['TRANSCRIBE_PROVIDER']        = 'local'
os.environ['WORKER_WORK_DIR']            = DRIVE_WORK_DIR

os.environ['ENABLE_RAILWAY_VERTICAL']         = 'false'
os.environ['ENABLE_PROXMOX_FULLRES']          = 'false'

print('Environment configured.')

### 🛠️ Optional: Manual Hotfix
If Git is too slow to push your changes, you can paste the content of `apps/worker/src/index.ts` below to manually patch the worker.

In [ ]:
# @title [HOTFIX] Patch index.ts manually
content = """\
// Paste content of apps/worker/src/index.ts here
"""

import os
target = '/content/CDV/apps/worker/src/index.ts'
if len(content) > 100:
    with open(target, 'w') as f:
        f.write(content)
    print(f"Patched {target} successfully.")
else:
    print("Cell skipped: No content provided.")

In [ ]:
# @title 6. Start Worker
import os
WORKER_PATH = '/content/CDV/apps/worker'
if os.path.exists(WORKER_PATH):
    %cd {WORKER_PATH}
    
    if not os.path.exists('node_modules'):
        print("\nnode_modules missing in worker folder! Attempting local install...")
        !pnpm install --no-frozen-lockfile
    
    print("\nStarting worker...")
    !pnpm run dev
else:
    print(f'ERROR: Worker path not found at {WORKER_PATH}')
    !ls -R /content/CDV